# Aula 2 — Redes neurais rasas (*shallow neural networks*)

**Deep Learning — PPGEE/UFMT** · Referência: S. Prince, *Understanding Deep Learning*, cap. 2 e 3.

**Roteiro da aula**

1. Revisão da aula 1: aprender = minimizar uma função de perda (regressão linear, treino e teste)
2. Problema 2.3: regressão linear discriminativa × generativa
3. O modelo de rede rasa (eq. 3.1) e a ReLU (eq. 3.2)
4. Por que a rede é não linear: no máximo 4 regiões lineares
5. Reconstruindo a figura 3.3: a não linearidade depende das unidades ocultas $h_d$
6. Só 3 inclinações são livres; articulações que coincidem
7. Unidades ocultas = capacidade; teorema da aproximação universal
8. Saídas multivariadas
9. Entradas multivariadas: planos, regiões poligonais convexas (figura 3.8)
10. O número de regiões cresce exponencialmente com a dimensão
11. Uma rede rasa geral: MNIST 20×20, grafo, terminologia e multiplicação de matrizes
12. **Exercício**: problema 3.9

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle
from matplotlib.colors import LinearSegmentedColormap
from math import comb

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

# cores usadas em toda a aula (as mesmas do livro: h1 laranja, h2 verde-água, h3 escuro)
C1, C2, C3 = '#D1603D', '#3E9E9A', '#2B2B2B'
CORES = [C1, C2, C3]
CMAP = LinearSegmentedColormap.from_list('livro', ['#2a1410', '#9c5a48', '#d99a80', '#f6dcc3'])

rng = np.random.default_rng(0)

---
## 1. Revisão da aula 1: aprender é otimizar

Um modelo é uma função $y = f[x, \boldsymbol\phi]$ com parâmetros $\boldsymbol\phi$. Na regressão linear 1D:

$$ y = f[x,\boldsymbol\phi] = \phi_0 + \phi_1 x. $$

Dado um conjunto de treino $\{x_i, y_i\}_{i=1}^{I}$, medimos o quão ruim é uma escolha de parâmetros com a **função de perda** (mínimos quadrados):

$$ L[\boldsymbol\phi] = \sum_{i=1}^{I}\big(f[x_i,\boldsymbol\phi]-y_i\big)^2 = \sum_{i=1}^{I}\big(\phi_0+\phi_1x_i-y_i\big)^2 .$$

**Aprender** = resolver o problema de otimização

$$ \hat{\boldsymbol\phi} = \underset{\boldsymbol\phi}{\operatorname{argmin}}\; L[\boldsymbol\phi]. $$

Depois, avaliamos o modelo em dados que **não** foram usados no treino (conjunto de teste). Toda a aula de hoje muda só o modelo $f$; a ideia de treinar minimizando $L$ continua a mesma.

In [ ]:
# Dados sintéticos: y = 0.3 + 0.5 x + ruído
N = 20
x_all = rng.uniform(0, 2, N)
y_all = 0.3 + 0.5 * x_all + rng.normal(0, 0.12, N)

# separação treino / teste
idx = rng.permutation(N)
x_tr, y_tr = x_all[idx[:14]], y_all[idx[:14]]
x_te, y_te = x_all[idx[14:]], y_all[idx[14:]]

def f_lin(x, phi):
    return phi[0] + phi[1] * x

def perda(phi, x, y):
    return np.sum((f_lin(x, phi) - y) ** 2)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(x_tr, y_tr, c=C1, label='treino')
ax.scatter(x_te, y_te, c=C2, marker='s', label='teste')
ax.set_xlabel('entrada, $x$'); ax.set_ylabel('saída, $y$'); ax.legend(); ax.set_title('Dados')
plt.show()

### O esquema de treinamento e teste

In [ ]:
def caixa(ax, xy, w, h, txt, cor):
    ax.add_patch(FancyBboxPatch(xy, w, h, boxstyle='round,pad=0.03', fc=cor, ec='k', lw=1))
    ax.text(xy[0] + w / 2, xy[1] + h / 2, txt, ha='center', va='center', fontsize=10)

def seta(ax, a, b):
    ax.annotate('', xy=b, xytext=a, arrowprops=dict(arrowstyle='->', lw=1.5))

fig, ax = plt.subplots(figsize=(11, 3.3))
ax.set_xlim(0, 10.2); ax.set_ylim(0, 3.2); ax.axis('off')
caixa(ax, (0.1, 1.1), 1.7, 1.0, 'Dados\n$\\{x_i, y_i\\}_{i=1}^{I}$', '#eeeeee')
caixa(ax, (2.5, 2.0), 1.7, 0.9, 'Treino\n(14 pontos)', '#fde0d6')
caixa(ax, (2.5, 0.3), 1.7, 0.9, 'Teste\n(6 pontos)', '#d6efee')
caixa(ax, (4.9, 2.0), 2.5, 0.9, 'Otimização\n$\\hat\\phi = \\mathrm{argmin}_\\phi\\, L[\\phi]$', '#fde0d6')
caixa(ax, (4.9, 0.3), 2.5, 0.9, 'Avaliação\n$L_{teste}[\\hat\\phi]$', '#d6efee')
caixa(ax, (8.2, 1.15), 1.9, 0.9, 'Desempenho em\ndados novos', '#eeeeee')
seta(ax, (1.8, 1.6), (2.5, 2.45)); seta(ax, (1.8, 1.6), (2.5, 0.75))
seta(ax, (4.2, 2.45), (4.9, 2.45)); seta(ax, (4.2, 0.75), (4.9, 0.75))
seta(ax, (6.15, 2.0), (6.15, 1.2)); ax.text(6.25, 1.55, r'$\hat\phi$', fontsize=13)
seta(ax, (7.4, 0.75), (8.2, 1.4))
ax.text(5.1, 3.05, 'os dados de teste NUNCA entram na otimização', fontsize=9, style='italic')
plt.show()

### Treinando com descida do gradiente

O gradiente da perda é (problema 2.1)

$$\frac{\partial L}{\partial \phi_0} = 2\sum_i(\phi_0+\phi_1x_i-y_i), \qquad \frac{\partial L}{\partial \phi_1} = 2\sum_i(\phi_0+\phi_1x_i-y_i)\,x_i ,$$

e a descida do gradiente repete $\boldsymbol\phi \leftarrow \boldsymbol\phi - \alpha\, \nabla L[\boldsymbol\phi]$.

In [ ]:
def grad(phi, x, y):
    r = f_lin(x, phi) - y
    return np.array([2 * r.sum(), 2 * (r * x).sum()])

phi = np.array([1.3, -0.6])          # chute inicial (ruim, de propósito)
alpha, T = 0.015, 100
hist = [phi.copy()]
for t in range(T):
    phi = phi - alpha * grad(phi, x_tr, y_tr)
    hist.append(phi.copy())
hist = np.array(hist)
phi_hat = hist[-1]
print('phi estimado =', phi_hat.round(3))

# superfície de perda (somente dados de TREINO)
p0, p1 = np.meshgrid(np.linspace(-0.6, 1.6, 200), np.linspace(-1.0, 1.6, 200))
Lgrid = ((p0[..., None] + p1[..., None] * x_tr - y_tr) ** 2).sum(-1)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
ax = axs[0]
cs = ax.contourf(p0, p1, np.log(Lgrid), levels=30, cmap=CMAP)
ax.contour(p0, p1, np.log(Lgrid), levels=15, colors='k', linewidths=0.3)
ax.plot(hist[:, 0], hist[:, 1], 'o-', c='cyan', ms=2, lw=1)
ax.plot(*hist[0], 'o', c='w', ms=7); ax.plot(*phi_hat, '*', c='yellow', ms=14)
ax.set_xlabel(r'intercepto $\phi_0$'); ax.set_ylabel(r'inclinação $\phi_1$')
ax.set_title('Perda $\\log L[\\phi]$ e caminho da descida do gradiente')

ax = axs[1]
xx = np.linspace(0, 2, 50)
for t, a in zip([0, 3, 10, 30, T], [0.2, 0.35, 0.5, 0.7, 1]):
    ax.plot(xx, f_lin(xx, hist[t]), c=C3, alpha=a, lw=1 + (t == T), label=f'iteração {t}')
ax.scatter(x_tr, y_tr, c=C1, zorder=3); ax.scatter(x_te, y_te, c=C2, marker='s', zorder=3)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.legend(fontsize=8); ax.set_title('O modelo durante o treino')
plt.tight_layout(); plt.show()

In [ ]:
# Curvas de treino e teste: usamos o erro MÉDIO para poder comparar conjuntos de tamanhos diferentes
mse_tr = [perda(p, x_tr, y_tr) / len(x_tr) for p in hist]
mse_te = [perda(p, x_te, y_te) / len(x_te) for p in hist]

fig, ax = plt.subplots(figsize=(6, 3.3))
ax.semilogy(mse_tr, c=C1, label='treino (usado na otimização)')
ax.semilogy(mse_te, c=C2, label='teste (só observado)')
ax.set_xlabel('iteração'); ax.set_ylabel('erro quadrático médio'); ax.legend()
plt.show()
print(f'MSE treino = {mse_tr[-1]:.4f}   MSE teste = {mse_te[-1]:.4f}')

---
## 2. Problema 2.3 — regressão linear generativa

**Enunciado (resumo).** Reformule a regressão linear como um modelo *generativo*, $x = g[y,\boldsymbol\phi] = \phi_0 + \phi_1 y$. (i) Qual é a nova função de perda? (ii) Qual a função inversa $y = g^{-1}[x,\boldsymbol\phi]$ usada na inferência? (iii) Este modelo faz as mesmas previsões que a versão discriminativa? Teste ajustando uma reta a três pontos pelos dois métodos.

**(i) Perda.** Agora o erro é medido em $x$:
$$L[\boldsymbol\phi] = \sum_i \big(\phi_0 + \phi_1 y_i - x_i\big)^2 .$$

**(ii) Inferência.** Invertendo: $\;y = g^{-1}[x,\boldsymbol\phi] = \dfrac{x-\phi_0}{\phi_1}.$

**(iii) Comparação.** Pelo problema 2.2, a solução de mínimos quadrados de $v \approx a + b\,u$ é
$b = \dfrac{\mathrm{cov}(u,v)}{\mathrm{var}(u)}$, $a = \bar v - b\,\bar u$. Logo

| | inclinação da reta $y(x)$ |
|---|---|
| discriminativo ($y$ a partir de $x$) | $\dfrac{\mathrm{cov}(x,y)}{\mathrm{var}(x)}$ |
| generativo invertido | $\dfrac{1}{\phi_1} = \dfrac{\mathrm{var}(y)}{\mathrm{cov}(x,y)}$ |

A razão entre as duas inclinações é $\rho^2 = \dfrac{\mathrm{cov}(x,y)^2}{\mathrm{var}(x)\,\mathrm{var}(y)}$. Elas só coincidem se $|\rho|=1$, isto é, se os pontos estão exatamente sobre uma reta. Em geral **as previsões são diferentes** (as duas retas se cruzam no ponto médio $(\bar x,\bar y)$).

Geometricamente: o modelo discriminativo minimiza os erros **verticais**; o generativo minimiza os erros **horizontais**.

In [ ]:
xp = np.array([0.3, 1.0, 1.7])
yp = np.array([0.5, 0.6, 1.4])

def minimos_quadrados(u, v):
    # ajusta v ≈ a + b u   (solução fechada do problema 2.2)
    b = np.sum((u - u.mean()) * (v - v.mean())) / np.sum((u - u.mean()) ** 2)
    return v.mean() - b * u.mean(), b

d0, d1 = minimos_quadrados(xp, yp)     # discriminativo: y = d0 + d1 x
g0, g1 = minimos_quadrados(yp, xp)     # generativo:     x = g0 + g1 y
y_disc = lambda x: d0 + d1 * x
y_gen = lambda x: (x - g0) / g1        # inversa usada na inferência

rho = np.corrcoef(xp, yp)[0, 1]
print(f'discriminativo: y = {d0:.3f} + {d1:.3f} x')
print(f'generativo:     x = {g0:.3f} + {g1:.3f} y   ->  y = (x {-g0:+.3f}) / {g1:.3f} = {-g0/g1:.3f} + {1/g1:.3f} x')
print(f'razão das inclinações = {d1 * g1:.4f}   rho^2 = {rho**2:.4f}')

xx = np.linspace(0, 2, 50)
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, f, nome, cor in [(axs[0], y_disc, 'Discriminativo: erros verticais', C1),
                         (axs[1], y_gen, 'Generativo: erros horizontais', C2)]:
    ax.plot(xx, f(xx), c=cor, lw=2)
    ax.scatter(xp, yp, c='k', zorder=3)
    for xi, yi in zip(xp, yp):
        if f is y_disc:
            ax.plot([xi, xi], [yi, f(xi)], 'k--', lw=1)
        else:
            ax.plot([xi, g0 + g1 * yi], [yi, yi], 'k--', lw=1)
    ax.set_title(nome); ax.set_xlabel('$x$'); ax.set_xlim(0, 2); ax.set_ylim(0, 1.8)
axs[0].set_ylabel('$y$')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(xx, y_disc(xx), c=C1, lw=2, label='discriminativo')
ax.plot(xx, y_gen(xx), c=C2, lw=2, label='generativo (invertido)')
ax.scatter(xp, yp, c='k', zorder=3); ax.plot(xp.mean(), yp.mean(), 'x', c='k', ms=10, label=r'$(\bar x,\bar y)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.legend(); ax.set_title('Previsões diferentes')
plt.show()

---
## 3. O modelo de rede rasa (eq. 3.1)

A regressão linear só descreve retas. Uma **rede neural rasa** com uma entrada $x$, uma saída $y$ e três unidades ocultas tem dez parâmetros
$\boldsymbol\phi = \{\phi_0,\phi_1,\phi_2,\phi_3,\theta_{10},\theta_{11},\theta_{20},\theta_{21},\theta_{30},\theta_{31}\}$:

$$
\boxed{\; y = f[x,\boldsymbol\phi] = \phi_0 + \phi_1\,\mathrm a[\theta_{10}+\theta_{11}x] + \phi_2\,\mathrm a[\theta_{20}+\theta_{21}x] + \phi_3\,\mathrm a[\theta_{30}+\theta_{31}x] \;} \qquad (3.1)
$$

O cálculo tem três passos:

1. calcula três funções lineares da entrada: $\theta_{10}+\theta_{11}x$, $\;\theta_{20}+\theta_{21}x$, $\;\theta_{30}+\theta_{31}x$;
2. passa cada uma pela **função de ativação** $\mathrm a[\bullet]$;
3. pondera os resultados por $\phi_1,\phi_2,\phi_3$, soma e adiciona o deslocamento $\phi_0$.

Chamando $h_d = \mathrm a[\theta_{d0}+\theta_{d1}x]$ de **unidades ocultas**:
$$ y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3 . \qquad (3.4)$$

### A ReLU (eq. 3.2)

A ativação mais comum é a *rectified linear unit*:
$$ \mathrm a[z] = \mathrm{ReLU}[z] = \begin{cases} 0 & z<0 \\ z & z\ge 0 \end{cases} \;=\; \max(0,z). $$

In [ ]:
def relu(z):
    return np.maximum(0, z)

z = np.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.plot(z, relu(z), c=C1, lw=2.5)
ax.axhline(0, c='gray', lw=0.5); ax.axvline(0, c='gray', lw=0.5)
ax.text(-2.8, 0.25, 'inativa\n(corta em zero)', fontsize=9)
ax.text(0.6, 2.0, 'ativa\n(passa $z$ adiante)', fontsize=9)
ax.set_xlabel('pré-ativação, $z$'); ax.set_ylabel(r'$\mathrm{ReLU}[z]$'); ax.set_title('ReLU')
plt.show()

In [ ]:
def rede_rasa_1d(x, phi, theta):
    '''Rede rasa 1D com D unidades ocultas.
    phi   = [phi_0, phi_1, ..., phi_D]          (D+1,)
    theta = [[th_10, th_11], ..., [th_D0, th_D1]] (D, 2)
    Retorna y, as ativações h (D, N) e as pré-ativações (D, N).'''
    pre = theta[:, [0]] + theta[:, [1]] * x[None, :]
    h = relu(pre)
    y = phi[0] + phi[1:] @ h
    return y, h, pre

def articulacoes(theta):
    # cada unidade "dobra" onde a sua pré-ativação é zero: x = -theta_d0 / theta_d1
    return -theta[:, 0] / theta[:, 1]

---
## 4. A rede é (potencialmente) não linear — e tem no máximo 4 regiões

Cada $h_d$ é uma reta **cortada** em zero. O corte acontece no ponto em que a pré-ativação se anula,
$$x_d = -\theta_{d0}/\theta_{d1},$$
que chamamos de **articulação** (*joint*). Entre duas articulações nenhuma unidade muda de estado, então $y$ é uma reta ali. Com 3 unidades há no máximo **3 articulações → no máximo 4 regiões lineares**. A função resultante é **linear por partes e contínua**.

Abaixo: seis redes com parâmetros sorteados. As linhas tracejadas são as articulações e as faixas alternadas são as regiões lineares dentro de $[0,2]$.

In [ ]:
def sombrear_regioes(ax, juntas, lo=0.0, hi=2.0, cor='0.85'):
    '''Sombreia regiões alternadas entre as articulações dentro de [lo, hi]. Retorna o nº de regiões.'''
    js = np.unique(np.round([j for j in juntas if lo < j < hi], 9))
    bordas = np.r_[lo, js, hi]
    for k in range(len(bordas) - 1):
        if k % 2 == 1:
            ax.axvspan(bordas[k], bordas[k + 1], color=cor, zorder=0)
    for j in js:
        ax.axvline(j, ls='--', c='gray', lw=0.8)
    return len(bordas) - 1

rng_r = np.random.default_rng(7)
x = np.linspace(0, 2, 400)
fig, axs = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)
for ax in axs.flat:
    # sorteia posição da articulação k e inclinação; theta_d0 = -theta_d1 * k
    while True:   # evita articulações quase coincidentes ou coladas na borda (ficariam invisíveis)
        k = rng_r.uniform(-0.3, 2.3, 3)
        pts = np.sort(np.r_[k, 0, 2])
        if np.diff(pts).min() > 0.15: break
    t1 = rng_r.normal(0, 1.5, 3)
    theta_r = np.c_[-t1 * k, t1]
    phi_r = rng_r.normal(0, 1, 4)
    y, _, _ = rede_rasa_1d(x, phi_r, theta_r)
    n = sombrear_regioes(ax, articulacoes(theta_r))
    ax.plot(x, y, c=C3, lw=2)
    ax.set_title(f'{n} regiões lineares em [0, 2]')
for ax in axs[1]: ax.set_xlabel('entrada, $x$')
plt.suptitle('Redes rasas aleatórias com 3 unidades ocultas: nunca mais que 4 regiões', y=1.0)
plt.tight_layout(); plt.show()

---
## 5. Reconstruindo a figura 3.3: a não linearidade vem das unidades ocultas

Vamos fixar parâmetros parecidos com os da figura 3.2a / 3.3 do livro e acompanhar o cálculo passo a passo:

- **(a–c)** as três retas $\theta_{d0}+\theta_{d1}x$;
- **(d–f)** cada reta passa pela ReLU: $h_d$ (o ponto onde corta é a articulação);
- **(g–i)** cada $h_d$ é escalado por $\phi_d$;
- **(j)** soma + $\phi_0$. Cada região linear corresponde a um **padrão de ativação** diferente: quais unidades estão ativas (1) e quais estão cortadas (0).

In [ ]:
theta = np.array([[-0.2,  0.6],     # h1: theta_10, theta_11
                  [-1.0,  1.0],     # h2: theta_20, theta_21
                  [ 0.8, -0.5]])    # h3: theta_30, theta_31
phi = np.array([0.0, -0.6, 0.8, 0.4])   # phi_0, phi_1, phi_2, phi_3

x = np.linspace(0, 2, 400)
y, h, pre = rede_rasa_1d(x, phi, theta)
juntas = articulacoes(theta)
print('articulações:', juntas.round(3))

fig = plt.figure(figsize=(11, 12))
gs = fig.add_gridspec(4, 3, height_ratios=[1, 1, 1, 1.3])
letras = 'abcdefghi'
for d in range(3):
    paineis = [(pre[d], rf'$\theta_{{{d+1}0}}+\theta_{{{d+1}1}}x$'),
               (h[d], rf'$h_{d+1}=\mathrm{{a}}[\theta_{{{d+1}0}}+\theta_{{{d+1}1}}x]$'),
               (phi[d + 1] * h[d], rf'$\phi_{d+1}h_{d+1}$')]
    for r, (curva, rot) in enumerate(paineis):
        ax = fig.add_subplot(gs[r, d])
        ax.plot(x, curva, c=CORES[d], lw=2)
        ax.axhline(0, c='gray', ls=':', lw=0.8)
        if r > 0:
            ax.axvline(juntas[d], c='gray', ls='--', lw=0.8)
        ax.set_ylim(-1, 1); ax.set_xlim(0, 2)
        ax.set_title(f'{letras[3*r+d]}) {rot}', fontsize=10)

ax = fig.add_subplot(gs[3, :])
ordem = np.sort(juntas); bordas = np.r_[0, ordem, 2]
for k in range(4):
    meio = (bordas[k] + bordas[k + 1]) / 2
    padrao = tuple(int(v) for v in (theta[:, 0] + theta[:, 1] * meio > 0))
    ax.axvspan(bordas[k], bordas[k + 1], color=['0.93', '0.8'][k % 2], zorder=0)
    ax.text(meio, 0.9, f'região {k+1}\n$(h_1,h_2,h_3)$ = {padrao}', ha='center', va='top', fontsize=9)
ax.plot(x, y, c=C3, lw=2.5)
ax.axhline(0, c='gray', ls=':', lw=0.8)
ax.set_ylim(-1, 1); ax.set_xlim(0, 2); ax.set_xlabel('entrada, $x$'); ax.set_ylabel('saída, $y$')
ax.set_title(r'j) $y=\phi_0+\phi_1h_1+\phi_2h_2+\phi_3h_3$')
plt.tight_layout(); plt.show()

Note como **a inclinação de cada região é a soma das inclinações das unidades ativas** nela. Por exemplo, na região 2 as unidades $h_1$ e $h_3$ estão ativas e $h_2$ está cortada, então a inclinação é $\theta_{11}\phi_1 + \theta_{31}\phi_3$ (a inclinação do painel g mais a do painel i).

### Explore: mova uma articulação

No Colab, use os controles abaixo. Mudar $\theta_{20}$ desloca a articulação de $h_2$; mudar $\phi_2$ muda quanto a inclinação "dobra" nela. Se $\phi_2 = 0$, a unidade $h_2$ deixa de contribuir e a função perde uma dobra.

In [ ]:
def explorar(theta_20=-1.0, theta_21=1.0, phi_2=0.8):
    th = theta.copy(); th[1] = [theta_20, theta_21]
    ph = phi.copy(); ph[2] = phi_2
    yy, _, _ = rede_rasa_1d(x, ph, th)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    n = sombrear_regioes(ax, articulacoes(th) if ph[1:].all() else articulacoes(th)[ph[1:] != 0])
    ax.plot(x, yy, c=C3, lw=2.5)
    ax.set_ylim(-1, 1); ax.set_xlim(0, 2); ax.set_title(f'{n} regiões lineares em [0, 2]')
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(explorar,
             theta_20=FloatSlider(value=-1.0, min=-2.5, max=0.5, step=0.05),
             theta_21=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.05),
             phi_2=FloatSlider(value=0.8, min=-1.5, max=1.5, step=0.05))
except ImportError:
    explorar()

---
## 6. A última frase da seção 3.1.1: só três inclinações são livres

> "Cada unidade oculta contribui com uma articulação, então com três unidades pode haver quatro regiões lineares. Porém, só três das inclinações dessas regiões são independentes; a quarta é zero (se todas as unidades estão inativas nessa região) ou é uma soma de inclinações das outras regiões."

Vamos destrinchar.

**Quantas regiões?** Cada articulação separa duas regiões. Se **duas articulações caem no mesmo ponto** (ou uma delas cai fora do intervalo de interesse), sobram apenas 2 pontos de dobra e portanto **uma região a menos**: 3 regiões.

**Quantas inclinações livres?** Seja $s_d = \phi_d\,\theta_{d1}$ a inclinação que a unidade $d$ acrescenta quando está ativa. A inclinação de uma região é a soma dos $s_d$ das unidades ativas nela.

Tome as articulações **em ordem**, da esquerda para a direita. Ao atravessar uma articulação, exatamente uma unidade troca de estado, então a inclinação muda de $\pm s_d$. As **quatro** inclinações são escritas com apenas **três** números livres $s_1, s_2, s_3$. Logo existe sempre uma relação entre elas: depois de fixar três regiões, a quarta inclinação já está determinada; não há um quarto parâmetro para escolhê-la.

Exemplo: se as três unidades estão ativas **à direita** de suas articulações ($\theta_{d1}>0$), então

| região | unidades ativas | inclinação |
|---|---|---|
| 1 | nenhuma | $0$ |
| 2 | uma | $s_a$ |
| 3 | duas | $s_a+s_b$ |
| 4 | três | $s_a+s_b+s_c$ |

As regiões 2, 3 e 4 têm inclinações livres; a região 1 é obrigada a ser plana. (A configuração da figura 3.3 é diferente: é o seu exercício no final da aula.)

**Contando graus de liberdade.** A função tem 3 posições de articulação + 3 mudanças de inclinação + 1 altura = **7** números efetivos, embora a rede tenha **10** parâmetros. Há redundância: por exemplo, $\mathrm{ReLU}[\alpha z] = \alpha\,\mathrm{ReLU}[z]$ para $\alpha>0$ (problemas 3.5–3.6).

In [ ]:
casos = {
    'figura 3.3: 4 regiões': (theta, phi),
    'h2 com a mesma articulação de h1: 3 regiões': (np.array([[-0.2, 0.6], [-1/3, 1.0], [0.8, -0.5]]), phi),
    'h3 com articulação fora de [0,2]: 3 regiões': (np.array([[-0.2, 0.6], [-1.0, 1.0], [1.5, -0.5]]), phi),
    'todas ativas à direita: região 1 plana': (np.array([[-0.2, 0.6], [-1.0, 1.0], [-0.8, 0.5]]),
                                               np.array([0.2, -0.9, 1.0, -0.9])),
}
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for ax, (nome, (th, ph)) in zip(axs.flat, casos.items()):
    yy, hh, _ = rede_rasa_1d(x, ph, th)
    n = sombrear_regioes(ax, articulacoes(th))
    for d in range(3):
        ax.plot(x, ph[d + 1] * hh[d], c=CORES[d], lw=1, alpha=0.6)
    ax.plot(x, yy, c=C3, lw=2.5)
    ax.set_ylim(-1, 1); ax.set_xlim(0, 2); ax.set_title(f'{nome}  (contadas: {n})', fontsize=10)
for ax in axs[1]: ax.set_xlabel('$x$')
plt.suptitle(r'linhas finas: $\phi_d h_d$;  linha grossa: $y$', y=1.0)
plt.tight_layout(); plt.show()

---
## 7. Unidades ocultas = capacidade; teorema da aproximação universal

Com $D$ unidades ocultas,
$$ h_d = \mathrm a[\theta_{d0}+\theta_{d1}x], \qquad y = \phi_0 + \sum_{d=1}^{D}\phi_d h_d ,$$
a função tem no máximo $D$ articulações e portanto no máximo $D+1$ regiões lineares. **O número de unidades ocultas mede a capacidade da rede**: quanto mais unidades, mais dobras, mais detalhe a rede consegue representar.

**Teorema da aproximação universal.** Para qualquer função contínua definida num subconjunto compacto de $\mathbb R^{D_i}$ e qualquer precisão desejada, existe uma rede rasa (com unidades ocultas suficientes) que aproxima a função com essa precisão.

**Ideia da prova em 1D (construtiva).** Coloque as articulações onde quiser, por exemplo igualmente espaçadas. Com os $\theta$ fixos, os $h_d$ são apenas **novas variáveis de entrada**, e encontrar os $\phi$ é uma **regressão linear** em $(1, h_1,\dots,h_D)$: exatamente o problema da aula 1! Mais articulações → regiões menores → a reta em cada região fica mais próxima da curva.

(Aqui a primeira articulação fica na borda esquerda, em $x=0$, só para dar a inclinação inicial; por isso $D$ unidades dão $D$ regiões dentro de $[0,2]$.)

In [ ]:
def alvo(x):
    return 0.55 * np.sin(5 * x + 0.6) + 0.25 * np.cos(9 * x)

def aproxima(x, alvos, D):
    k = np.linspace(0, 2, D, endpoint=False)        # articulações
    theta_u = np.c_[-k, np.ones(D)]                   # h_d = ReLU(x - k_d)
    H = relu(theta_u[:, [0]] + theta_u[:, [1]] * x[None, :]).T   # (N, D)
    A = np.c_[np.ones(len(x)), H]                     # colunas: 1, h_1, ..., h_D
    PHI, *_ = np.linalg.lstsq(A, alvos, rcond=None)   # regressão linear!
    return A @ PHI, k

x = np.linspace(0, 2, 600)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, D, cor in zip(axs, [5, 10, 20], ['0.5', C1, C2]):
    yh, k = aproxima(x, alvo(x), D)
    ax.plot(x, alvo(x), 'k--', lw=1.3, label='função alvo')
    ax.plot(x, yh, c=cor, lw=2.5, alpha=0.9, label=f'rede com D={D}')
    for kk in k[1:]: ax.axvline(kk, c='gray', lw=0.4)
    ax.set_title(f'{D} regiões lineares'); ax.set_xlabel('entrada, $x$'); ax.set_ylim(-1, 1)
axs[0].set_ylabel('saída, $y$'); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

Ds = np.array([2, 3, 5, 8, 12, 20, 30, 50, 80, 120])
erros = [np.mean((aproxima(x, alvo(x), D)[0] - alvo(x)) ** 2) for D in Ds]
fig, ax = plt.subplots(figsize=(5, 3.3))
ax.loglog(Ds, erros, 'o-', c=C3)
ax.set_xlabel('unidades ocultas, $D$'); ax.set_ylabel('erro quadrático médio')
ax.set_title('Mais capacidade, menor erro de aproximação')
plt.show()

---
## 8. Saídas multivariadas

Para $D_o$ saídas, cada saída é **outra combinação linear das mesmas unidades ocultas**. Com 4 unidades e 2 saídas:
$$
\begin{aligned}
h_d &= \mathrm a[\theta_{d0}+\theta_{d1}x], \quad d=1,\dots,4\\
y_1 &= \phi_{10} + \phi_{11}h_1 + \phi_{12}h_2 + \phi_{13}h_3 + \phi_{14}h_4\\
y_2 &= \phi_{20} + \phi_{21}h_1 + \phi_{22}h_2 + \phi_{23}h_3 + \phi_{24}h_4
\end{aligned}
$$

Consequência: **as articulações são as mesmas para todas as saídas** (elas dependem só dos $\theta$). O que muda de uma saída para outra são as inclinações e alturas (os $\phi$).

In [ ]:
theta4 = np.array([[-0.3, 1.0], [-0.8, 1.0], [1.3, -1.0], [-1.6, 1.0]])
Phi = np.array([[ 0.1,  1.0, -2.0,  0.5,  2.0],      # y1: phi_10, ..., phi_14
                [-0.4, -0.8,  1.5,  0.6, -1.0]])     # y2: phi_20, ..., phi_24
x = np.linspace(0, 2, 400)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for i, ax in enumerate(axs):
    yy, _, _ = rede_rasa_1d(x, Phi[i], theta4)
    sombrear_regioes(ax, articulacoes(theta4))
    ax.plot(x, yy, c=[C1, C2][i], lw=2.5)
    ax.set_title(f'saída $y_{i+1}$'); ax.set_xlabel('$x$'); ax.set_xlim(0, 2)
plt.suptitle('Mesmas articulações (linhas tracejadas), funções diferentes', y=1.02)
plt.tight_layout(); plt.show()

**Isso limita o que a rede aprende?** Não, desde que haja unidades suficientes. Abaixo, uma única camada oculta com $D=12$ articulações fixas aproxima **duas funções completamente diferentes** ao mesmo tempo. Como na seção 7, os pesos de saída vêm de uma regressão linear, agora com uma coluna de alvos por saída.

In [ ]:
def alvo2(x):
    return 0.8 * np.exp(-6 * (x - 1.2) ** 2) - 0.4 * x + 0.2

alvos = np.c_[alvo(x), alvo2(x)]           # (N, 2): uma coluna por saída
Yh, k = aproxima(x, alvos, 12)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for i, ax in enumerate(axs):
    for kk in k[1:]: ax.axvline(kk, c='gray', lw=0.4)
    ax.plot(x, alvos[:, i], 'k--', lw=1.3)
    ax.plot(x, Yh[:, i], c=[C1, C2][i], lw=2.5, alpha=0.9)
    ax.set_title(f'saída $y_{i+1}$ (D = 12 unidades compartilhadas)'); ax.set_xlabel('$x$')
plt.tight_layout(); plt.show()

---
## 9. Entradas multivariadas: cada unidade oculta é um plano

Com duas entradas $\mathbf x = [x_1, x_2]^T$:
$$ h_d = \mathrm a[\theta_{d0} + \theta_{d1}x_1 + \theta_{d2}x_2], \qquad y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3 .$$

Agora a pré-ativação $\theta_{d0}+\theta_{d1}x_1+\theta_{d2}x_2$ é um **plano** sobre o espaço $(x_1,x_2)$. A ReLU corta o plano ao longo da reta onde ele cruza zero; essa reta faz o papel da "articulação".

Primeiro, um plano isolado (figura 3.8a). Os mapas de cor que vamos usar são a **vista de cima** desse plano: o brilho é a altura e as linhas finas são curvas de nível.

In [ ]:
theta2 = np.array([[ 0.2,  1.0,  0.6],     # theta_10, theta_11, theta_12
                   [-0.3,  1.0, -0.15],    # theta_20, theta_21, theta_22
                   [-0.3, -0.15, 1.0]])    # theta_30, theta_31, theta_32
phi2 = np.array([0.0, -1.0, 1.5, -0.8])

g = np.linspace(-1, 1, 301)
X1, X2 = np.meshgrid(g, g)
PRE = [theta2[d, 0] + theta2[d, 1] * X1 + theta2[d, 2] * X2 for d in range(3)]
H2 = [relu(P) for P in PRE]
Y2 = phi2[0] + sum(phi2[d + 1] * H2[d] for d in range(3))

def mapa(ax, Z, titulo, juntas=()):
    ax.imshow(Z, extent=[-1, 1, -1, 1], origin='lower', cmap=CMAP, aspect='equal')
    if np.ptp(Z) > 0:
        ax.contour(X1, X2, Z, levels=20, colors='k', linewidths=0.3)
    for P in juntas:
        ax.contour(X1, X2, P, levels=[0], colors='cyan', linewidths=2)
    ax.set_title(titulo, fontsize=9); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig = plt.figure(figsize=(11, 4.3))
ax = fig.add_subplot(1, 2, 1, projection='3d')
s = slice(None, None, 6)
ax.plot_surface(X1[s, s], X2[s, s], PRE[0][s, s], cmap=CMAP, alpha=0.95, linewidth=0)
zmin = PRE[0].min() - 0.8
ax.contourf(X1, X2, PRE[0], levels=20, zdir='z', offset=zmin, cmap=CMAP)
ax.set_zlim(zmin, PRE[0].max()); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title(r'plano $\theta_{10}+\theta_{11}x_1+\theta_{12}x_2$' + '\n(embaixo: a sua projeção = vista de cima)')
ax.view_init(25, -60)
mapa(fig.add_subplot(1, 2, 2), PRE[0], r'a) vista de cima do mesmo plano')
plt.tight_layout(); plt.show()

### Reconstruindo a figura 3.8

- **(a–c)** os três planos (pré-ativações);
- **(d–f)** cada plano cortado pela ReLU; a **linha ciano** é a articulação (onde o plano cruza zero);
- **(g–i)** cada $h_d$ ponderado por $\phi_d$;
- **(j)** soma + $\phi_0$: uma superfície contínua formada por **regiões poligonais convexas**.

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(11, 11))
for d in range(3):
    i, j = d + 1, d + 1
    mapa(axs[0, d], PRE[d], f'{"abc"[d]}) ' + rf'$\theta_{{{i}0}}+\theta_{{{i}1}}x_1+\theta_{{{i}2}}x_2$')
    mapa(axs[1, d], H2[d], f'{"def"[d]}) ' + rf'$h_{i}=\mathrm{{a}}[\theta_{{{i}0}}+\theta_{{{i}1}}x_1+\theta_{{{i}2}}x_2]$', [PRE[d]])
    mapa(axs[2, d], phi2[d + 1] * H2[d], f'{"ghi"[d]}) ' + rf'$\phi_{i}h_{i}$', [PRE[d]])
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
mapa(ax, Y2, r'j) $y=\phi_0+\phi_1h_1+\phi_2h_2+\phi_3h_3$', PRE)
plt.show()

### O mesmo resultado em 3D

Cada mapa é a vista de cima de uma superfície. O $h_1$ cortado é um plano "dobrado" na linha ciano; a saída $y$ é a soma de três planos dobrados, ou seja, uma superfície feita de **pedaços de planos** colados nas linhas ciano.

In [ ]:
def linha_articulacao(d, n=200):
    # pontos (x1, x2) dentro de [-1,1]^2 onde theta_d0 + theta_d1 x1 + theta_d2 x2 = 0
    nrm = theta2[d, 1:]; p0 = -theta2[d, 0] * nrm / (nrm @ nrm)
    t = np.linspace(-3, 3, 4000); pts = p0[None] + t[:, None] * np.array([-nrm[1], nrm[0]])[None]
    return pts[(np.abs(pts) <= 1).all(1)]

def y_rede2(p):
    hh = relu(theta2[:, 0][None] + p @ theta2[:, 1:].T)
    return phi2[0] + hh @ phi2[1:]

fig = plt.figure(figsize=(15, 5))
paineis = [(PRE[0], 'a) plano (pré-ativação de $h_1$)', []),
           (H2[0], 'd) $h_1$: plano cortado pela ReLU', [0]),
           (Y2, 'j) $y$: combinação de planos dobrados', [0, 1, 2])]
for i, (Z, tit, ds) in enumerate(paineis):
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    zmin = Z.min() - 0.6
    ax.plot_surface(X1[s, s], X2[s, s], Z[s, s], cmap=CMAP, alpha=0.9, linewidth=0)
    ax.contourf(X1, X2, Z, levels=20, zdir='z', offset=zmin, cmap=CMAP)
    for d in ds:
        pts = linha_articulacao(d)
        zz = y_rede2(pts) if Z is Y2 else relu(theta2[d, 0] + pts @ theta2[d, 1:])
        ax.plot(pts[:, 0], pts[:, 1], zz, c='cyan', lw=2.5)
        ax.contour(X1, X2, PRE[d], levels=[0], zdir='z', offset=zmin, colors='cyan')
    ax.set_zlim(zmin, Z.max() + 0.1); ax.set_title(tit); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.view_init(32, -125 if i == 2 else -55)
plt.tight_layout(); plt.show()

### Regiões poligonais convexas e padrões de ativação

Dentro de cada região, o padrão de ativação $(h_1,h_2,h_3)$ é constante, então $y$ é um **único plano** ali. Cada região é a interseção de semiplanos (um por unidade: "ativa" ou "inativa"), e **interseção de semiplanos é sempre um polígono convexo**.

In [ ]:
padrao = sum((PRE[d] > 0).astype(int) << d for d in range(3))
codigos = np.unique(padrao)
print('número de regiões:', len(codigos))

fig, ax = plt.subplots(figsize=(5.5, 5.5))
lut = {c: i for i, c in enumerate(codigos)}
ax.imshow(np.vectorize(lut.get)(padrao), extent=[-1, 1, -1, 1], origin='lower',
          cmap='Pastel1', vmin=0, vmax=8)
for P in PRE:
    ax.contour(X1, X2, P, levels=[0], colors='cyan', linewidths=2)
for c in codigos:
    m = padrao == c
    bits = tuple(int((c >> d) & 1) for d in range(3))
    ax.text(X1[m].mean(), X2[m].mean(), str(bits), ha='center', va='center', fontsize=9)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title(f'{len(codigos)} regiões convexas; rótulo = $(h_1,h_2,h_3)$ ativos')
plt.show()

---
## 10. O número de regiões cresce exponencialmente

Em 1D, $D$ unidades dão no máximo $D+1$ regiões. Em $D_i$ dimensões, cada unidade é um **hiperplano** que corta o espaço em dois, e o número máximo de regiões criadas por $D$ hiperplanos é (Zaslavsky, 1975)

$$ N_{\max}(D, D_i) = \sum_{j=0}^{D_i}\binom{D}{j}. $$

- $D_i=1$: $1 + D$ (o caso 1D).
- $D_i=2$, $D=3$: $1+3+3=7$, as 7 regiões da figura 3.8.
- Se $D_i \ge D$: $N_{\max} = 2^D$; cada combinação de ativas/inativas vira uma região.

O número de parâmetros cresce **linearmente** com $D$, mas o número de regiões cresce como um polinômio de grau $D_i$ em $D$ e chega a $2^D$. A mesma rede, com o mesmo número de unidades, divide o espaço em muito mais pedaços quando a entrada tem mais dimensões.

In [ ]:
def max_regioes(D, Di):
    return sum(comb(D, j) for j in range(0, min(D, Di) + 1))

print('D=3, Di=2 :', max_regioes(3, 2))
print('D=5, Di=2 :', max_regioes(5, 2))
print('D=10, Di=10:', max_regioes(10, 10), '= 2^10')

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
Ds = np.arange(1, 101)
for Di, cor in zip([1, 2, 5, 10], ['0.6', C1, C2, C3]):
    axs[0].semilogy(Ds, [max_regioes(D, Di) for D in Ds], c=cor, lw=2, label=f'$D_i$ = {Di}')
axs[0].set_xlabel('unidades ocultas, $D$'); axs[0].set_ylabel('máximo de regiões')
axs[0].legend(); axs[0].set_title('Regiões × unidades ocultas')

Dis = np.arange(1, 21)
for D, cor in zip([5, 10, 15], [C1, C2, C3]):
    axs[1].semilogy(Dis, [max_regioes(D, Di) for Di in Dis], 'o-', c=cor, ms=3, label=f'D = {D}  (teto $2^{{{D}}}$)')
axs[1].set_xlabel('dimensão da entrada, $D_i$'); axs[1].set_ylabel('máximo de regiões')
axs[1].legend(); axs[1].set_title('Regiões × dimensão da entrada')
plt.tight_layout(); plt.show()

In [ ]:
# A mesma quantidade de unidades (D = 5) em 1D e em 2D
rng_v = np.random.default_rng(3)
th1d = np.c_[-(np.array([0.3, 0.65, 1.0, 1.35, 1.7]) + rng_v.uniform(-0.1, 0.1, 5)), np.ones(5)]   # 5 articulações em [0, 2]
ang = rng_v.uniform(0, np.pi, 5); pts = rng_v.uniform(-0.6, 0.6, (5, 2))
nrm = np.c_[np.cos(ang), np.sin(ang)]
th2d = np.c_[-(nrm * pts).sum(1), nrm]                          # 5 retas passando perto do centro

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [1.4, 1]})
n1 = sombrear_regioes(axs[0], articulacoes(th1d))
axs[0].set_xlim(0, 2); axs[0].set_yticks([]); axs[0].set_xlabel('$x$')
axs[0].set_title(f'1D, D = 5: {n1} regiões (máx. {max_regioes(5, 1)})')

PP = [th2d[d, 0] + th2d[d, 1] * X1 + th2d[d, 2] * X2 for d in range(5)]
pd = sum((P > 0).astype(int) << d for d, P in enumerate(PP))
u, inv = np.unique(pd, return_inverse=True)
axs[1].imshow(inv.reshape(pd.shape), extent=[-1, 1, -1, 1], origin='lower', cmap='tab20')
for P in PP: axs[1].contour(X1, X2, P, levels=[0], colors='k', linewidths=1)
axs[1].set_title(f'2D, D = 5: {len(u)} regiões no quadrado (máx. {max_regioes(5, 2)})')
axs[1].set_xlabel('$x_1$'); axs[1].set_ylabel('$x_2$')
plt.tight_layout(); plt.show()

---
## 11. Uma rede rasa geral: classificando dígitos do MNIST

A rede rasa geral (eqs. 3.11–3.12) mapeia $\mathbf x\in\mathbb R^{D_i}$ em $\mathbf y \in \mathbb R^{D_o}$ com $D$ unidades ocultas:

$$ h_d = \mathrm a\Big[\theta_{d0} + \sum_{i=1}^{D_i}\theta_{di}x_i\Big], \qquad y_j = \phi_{j0} + \sum_{d=1}^{D}\phi_{jd}h_d .$$

Vamos usar imagens de dígitos de **20×20 pixels**. Cada imagem vira um vetor com $D_i = 400$ entradas, a rede tem $D = 64$ unidades ocultas e $D_o = 10$ saídas (uma por dígito; a maior indica a classe prevista).

Curiosidade: no MNIST original, cada dígito foi normalizado para caber numa caixa de 20×20 e depois centralizado numa imagem 28×28. Então basta recortar o centro.

### O grafo da rede e a terminologia

In [ ]:
def desenha_rede(ax, Di=400, D=64, Do=10):
    camadas = [(0.0, ['$x_1$', '$x_2$', '$x_3$', '...', f'$x_{{{Di}}}$']),
               (3.2, ['$h_1$', '$h_2$', '$h_3$', '...', f'$h_{{{D}}}$']),
               (6.4, ['$y_1$', '$y_2$', '...', f'$y_{{{Do}}}$'])]
    pos = []
    for X, nomes in camadas:
        ys = np.linspace(len(nomes) - 1, 0, len(nomes)) * 0.9 - (len(nomes) - 1) * 0.45
        pos.append([(X, yy, n) for yy, n in zip(ys, nomes)])
    vieses = [(0.0, 2.9), (3.2, 2.9)]
    # arestas
    for l in range(2):
        for (xa, ya, na) in pos[l]:
            for (xb, yb, nb) in pos[l + 1]:
                if na != '...' and nb != '...':
                    ax.plot([xa, xb], [ya, yb], c='0.6', lw=0.6, zorder=1)
        for (xb, yb, nb) in pos[l + 1]:
            if nb != '...':
                ax.plot([vieses[l][0], xb], [vieses[l][1], yb], c=C1, lw=0.8, zorder=1)
    # nós
    for camada in pos:
        for (X, Y, n) in camada:
            if n == '...':
                ax.text(X, Y, r'$\vdots$', ha='center', va='center', fontsize=16)
            else:
                ax.add_patch(Circle((X, Y), 0.32, fc='white', ec='k', zorder=2))
                ax.text(X, Y, n, ha='center', va='center', fontsize=10, zorder=3)
    for (X, Y) in vieses:
        ax.add_patch(Circle((X, Y), 0.28, fc='#fde0d6', ec=C1, zorder=2))
        ax.text(X, Y, '1', ha='center', va='center', zorder=3)
    # rótulos
    ax.text(0.0, -3.0, f'camada de entrada\n$D_i$ = {Di}', ha='center')
    ax.text(3.2, -3.0, f'camada oculta\n$D$ = {D} unidades', ha='center')
    ax.text(6.4, -3.0, f'camada de saída\n$D_o$ = {Do}', ha='center')
    ax.text(1.6, 2.45, r'pesos $\Theta$' + f'\n({D}×{Di})', ha='center', fontsize=9)
    ax.text(4.8, 2.45, r'pesos $\Phi$' + f'\n({Do}×{D})', ha='center', fontsize=9)
    ax.text(-0.9, 2.9, r'vieses $\theta_{\bullet 0}$', ha='right', va='center', fontsize=9, color=C1)
    ax.text(2.3, 3.4, r'vieses $\phi_{\bullet 0}$', ha='center', va='center', fontsize=9, color=C1)
    ax.set_xlim(-2.2, 7.6); ax.set_ylim(-3.6, 3.6); ax.set_aspect('equal'); ax.axis('off')

fig, ax = plt.subplots(figsize=(9, 6))
desenha_rede(ax)
plt.show()

**Terminologia**

| termo | significado |
|---|---|
| camada de entrada / oculta / de saída | as três "colunas" do grafo |
| unidade oculta (neurônio) | cada $h_d$ |
| **pré-ativação** | o valor $\theta_{d0}+\sum_i\theta_{di}x_i$ antes da ReLU |
| **ativação** | o valor $h_d$ depois da ReLU |
| **pesos** | os coeficientes que multiplicam entradas ($\theta_{di}$, $\phi_{jd}$): as arestas cinzas |
| **vieses** (*biases*) | os termos constantes $\theta_{d0}$, $\phi_{j0}$: as arestas laranja, vindas de um nó fixo igual a 1 |
| totalmente conectada | toda unidade de uma camada liga em toda unidade da seguinte |
| largura | o número de unidades ocultas $D$ (a capacidade da rede rasa) |
| rede rasa | exatamente **uma** camada oculta (redes profundas: cap. 4) |

### Tudo é multiplicação de matrizes

Cada linha do sistema abaixo é uma unidade oculta:

$$
\begin{aligned}
h_1 &= \mathrm a[\theta_{1,0} + \theta_{1,1}x_1 + \theta_{1,2}x_2 + \cdots + \theta_{1,400}x_{400}]\\
h_2 &= \mathrm a[\theta_{2,0} + \theta_{2,1}x_1 + \theta_{2,2}x_2 + \cdots + \theta_{2,400}x_{400}]\\
&\;\;\vdots\\
h_{64} &= \mathrm a[\theta_{64,0} + \theta_{64,1}x_1 + \theta_{64,2}x_2 + \cdots + \theta_{64,400}x_{400}]
\end{aligned}
$$

Empilhando, os coeficientes formam uma **matriz** e as entradas, um **vetor**:

$$
\boxed{\begin{bmatrix} h_1\\ h_2\\ \vdots\\ h_{64}\end{bmatrix}}
= \mathrm a\left[\;
\boxed{\begin{bmatrix} \theta_{1,0}\\ \theta_{2,0}\\ \vdots\\ \theta_{64,0}\end{bmatrix}}
+
\boxed{\begin{bmatrix}
\theta_{1,1} & \theta_{1,2} & \cdots & \theta_{1,400}\\
\theta_{2,1} & \theta_{2,2} & \cdots & \theta_{2,400}\\
\vdots & \vdots & \ddots & \vdots\\
\theta_{64,1} & \theta_{64,2} & \cdots & \theta_{64,400}
\end{bmatrix}}
\boxed{\begin{bmatrix} x_1\\ x_2\\ \vdots\\ x_{400}\end{bmatrix}}
\;\right]
\quad\Longleftrightarrow\quad
\mathbf h = \mathrm a[\boldsymbol\theta_0 + \boldsymbol\Theta\mathbf x]
$$

$$
\boxed{\begin{bmatrix} y_1\\ \vdots\\ y_{10}\end{bmatrix}}
=
\boxed{\begin{bmatrix} \phi_{1,0}\\ \vdots\\ \phi_{10,0}\end{bmatrix}}
+
\boxed{\begin{bmatrix}
\phi_{1,1} & \cdots & \phi_{1,64}\\
\vdots & \ddots & \vdots\\
\phi_{10,1} & \cdots & \phi_{10,64}
\end{bmatrix}}
\boxed{\begin{bmatrix} h_1\\ \vdots\\ h_{64}\end{bmatrix}}
\quad\Longleftrightarrow\quad
\mathbf y = \boldsymbol\phi_0 + \boldsymbol\Phi\mathbf h
$$

A figura abaixo desenha as matrizes **em escala** (cada lado é proporcional ao número de linhas/colunas). Em destaque: a linha 1 de $\boldsymbol\Theta$ vezes o vetor $\mathbf x$ dá a pré-ativação de $h_1$.

In [ ]:
def bloco(ax, x0, w, h, rot, cor, dim, destaque=None):
    ax.add_patch(Rectangle((x0, -h / 2), w, h, fc=cor, ec='k', lw=1.2))
    if destaque is not None:          # (altura da faixa) — destaca a primeira linha
        ax.add_patch(Rectangle((x0, h / 2 - destaque), w, destaque, fc='yellow', ec='k', lw=0.8, alpha=0.9))
    ax.text(x0 + w / 2, 0, rot, ha='center', va='center', fontsize=13)
    ax.text(x0 + w / 2, -h / 2 - 0.25, dim, ha='center', va='top', fontsize=9)
    return x0 + w

def simbolo(ax, x0, s, dx=0.35, fs=16):
    ax.text(x0 + dx / 2, 0, s, ha='center', va='center', fontsize=fs)
    return x0 + dx

esc, col, lin = 0.011, 0.28, 0.07        # escala: unidades -> polegadas de desenho
Di, D, Do = 400, 64, 10
fig, axs = plt.subplots(2, 1, figsize=(12, 8.5), gridspec_kw={'height_ratios': [1.25, 0.6]})

ax = axs[0]; p = 0
p = bloco(ax, p, col, D * esc, r'$\mathbf{h}$', '#d6efee', f'{D}×1', destaque=lin)
p = simbolo(ax, p, '=')
p = simbolo(ax, p, 'a[', 0.6, fs=34)
p = bloco(ax, p, col, D * esc, r'$\theta_0$', '#fde0d6', f'{D}×1', destaque=lin)
p = simbolo(ax, p, '+')
p = bloco(ax, p, Di * esc, D * esc, r'$\Theta$', '#fde0d6', f'{D}×{Di}', destaque=lin)
p = simbolo(ax, p, ' ', 0.15)
p = bloco(ax, p, col, Di * esc, r'$\mathbf{x}$', '#eeeeee', f'{Di}×1')
ax.add_patch(Rectangle((p - col, -Di * esc / 2), col, Di * esc, fc='yellow', ec='k', alpha=0.35))
p = simbolo(ax, p, ']', 0.35, fs=34)
ax.annotate(r'linha 1 de $\Theta$ · $\mathbf{x}$  +  $\theta_{1,0}$  →  pré-ativação de $h_1$',
            xy=(2.6, D * esc / 2 - lin / 2), xytext=(1.2, 1.9), fontsize=10,
            arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.3, p + 0.3); ax.set_ylim(-2.6, 2.4); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Camada oculta', loc='left')

ax = axs[1]; p = 0
p = bloco(ax, p, col, Do * esc, r'$\mathbf{y}$', '#eeeeee', f'{Do}×1')
p = simbolo(ax, p, '=')
p = bloco(ax, p, col, Do * esc, r'$\phi_0$', '#d6efee', f'{Do}×1')
p = simbolo(ax, p, '+')
p = bloco(ax, p, D * esc, Do * esc, r'$\Phi$', '#d6efee', f'{Do}×{D}')
p = simbolo(ax, p, ' ', 0.15)
p = bloco(ax, p, col, D * esc, r'$\mathbf{h}$', '#d6efee', f'{D}×1')
ax.set_xlim(-0.3, 6.6); ax.set_ylim(-0.8, 0.6); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Camada de saída', loc='left')
plt.tight_layout(); plt.show()

n_par = D * (Di + 1) + Do * (D + 1)
print(f'parâmetros: D(Di+1) + Do(D+1) = {D}·{Di+1} + {Do}·{D+1} = {n_par}')

### Os dados

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets

treino = datasets.MNIST('dados', train=True, download=True)
teste = datasets.MNIST('dados', train=False, download=True)

def prepara(ds):
    X = ds.data[:, 4:24, 4:24].float() / 255.0     # recorta o centro 20×20
    return X.reshape(len(X), -1), ds.targets        # cada imagem vira um vetor de 400

X_tr, Y_tr = prepara(treino)
X_te, Y_te = prepara(teste)
print('treino:', X_tr.shape, '  teste:', X_te.shape)

fig, axs = plt.subplots(2, 8, figsize=(11, 3))
for ax, i in zip(axs.flat, range(16)):
    ax.imshow(X_tr[i].reshape(20, 20), cmap='gray_r'); ax.set_title(int(Y_tr[i])); ax.axis('off')
plt.suptitle('Imagens 20×20 → vetores $\\mathbf{x}\\in\\mathbb{R}^{400}$'); plt.tight_layout(); plt.show()

### O modelo, escrito exatamente como as equações

Escrevemos os quatro tensores de parâmetros à mão, com os nomes do livro. Em código processamos um **lote** de imagens de uma vez: cada linha de `X` é um $\mathbf x^T$, por isso aparece `X @ Theta.T`; para uma única imagem é exatamente $\boldsymbol\theta_0 + \boldsymbol\Theta\mathbf x$.

In [ ]:
torch.manual_seed(0)
Di, D, Do = 400, 64, 10

Theta0 = torch.zeros(D, requires_grad=True)                                  # θ0: (64,)
Theta  = (torch.randn(D, Di) * np.sqrt(2 / Di)).requires_grad_()              # Θ : (64, 400)
Phi0   = torch.zeros(Do, requires_grad=True)                                 # φ0: (10,)
Phi    = (torch.randn(Do, D) * np.sqrt(2 / D)).requires_grad_()               # Φ : (10, 64)
parametros = [Theta0, Theta, Phi0, Phi]

def rede(X):
    H = torch.relu(Theta0 + X @ Theta.T)     # h = a[θ0 + Θx]   -> (B, 64)
    return Phi0 + H @ Phi.T                  # y = φ0 + Φh      -> (B, 10)

# uma única imagem, na forma "de livro"
x1 = X_tr[0]
h1 = torch.relu(Theta0 + Theta @ x1)
y1 = Phi0 + Phi @ h1
print('x:', tuple(x1.shape), ' h:', tuple(h1.shape), ' y:', tuple(y1.shape))
print('número de parâmetros:', sum(p.numel() for p in parametros))

### Treino e teste

O esquema é o mesmo da seção 1: minimizamos a perda **só nos dados de treino** e medimos o desempenho no teste. A perda para classificação (entropia cruzada) e o otimizador serão estudados nos capítulos 5 e 6; por enquanto, pense nelas como "a $L[\boldsymbol\phi]$ certa para classes" e "uma descida do gradiente melhorada".

In [ ]:
otimizador = torch.optim.SGD(parametros, lr=0.05, momentum=0.9)
hist_tr, hist_te = [], []

def avalia(X, Y):
    with torch.no_grad():
        saida = rede(X)
        return F.cross_entropy(saida, Y).item(), (saida.argmax(1) == Y).float().mean().item()

for epoca in range(6):
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), 128):
        lote = perm[i:i + 128]
        L = F.cross_entropy(rede(X_tr[lote]), Y_tr[lote])
        otimizador.zero_grad(); L.backward(); otimizador.step()
    hist_tr.append(avalia(X_tr, Y_tr)); hist_te.append(avalia(X_te, Y_te))
    print(f'época {epoca+1}: perda treino {hist_tr[-1][0]:.3f} | acurácia treino {hist_tr[-1][1]:.3f}'
          f' | acurácia teste {hist_te[-1][1]:.3f}')

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(range(1, 7), [h[0] for h in hist_tr], 'o-', c=C1, label='treino')
ax.plot(range(1, 7), [h[0] for h in hist_te], 's-', c=C2, label='teste')
ax.set_xlabel('época'); ax.set_ylabel('perda'); ax.legend(); plt.show()

### O que as unidades ocultas aprenderam

Cada linha de $\boldsymbol\Theta$ tem 400 números: podemos vê-la como uma imagem 20×20. A pré-ativação de $h_d$ é o produto interno dessa "imagem-filtro" com a entrada. Geometricamente, cada unidade continua sendo um **hiperplano** (agora em $\mathbb R^{400}$) que divide o espaço das imagens em "ativa" e "inativa", exatamente como as retas ciano da seção 9.

In [ ]:
W = Theta.detach()
fig, axs = plt.subplots(2, 8, figsize=(11, 3))
for d, ax in enumerate(axs.flat):
    m = W[d].abs().max()
    ax.imshow(W[d].reshape(20, 20), cmap='RdBu_r', vmin=-m, vmax=m); ax.axis('off'); ax.set_title(f'$h_{{{d+1}}}$')
plt.suptitle(r'Linhas de $\Theta$ (vermelho: peso positivo, azul: negativo)'); plt.tight_layout(); plt.show()

with torch.no_grad():
    prev = rede(X_te[:12]).argmax(1)
fig, axs = plt.subplots(1, 12, figsize=(12, 1.6))
for i, ax in enumerate(axs):
    ax.imshow(X_te[i].reshape(20, 20), cmap='gray_r'); ax.axis('off')
    ax.set_title(int(prev[i]), color='k' if prev[i] == Y_te[i] else 'r')
plt.suptitle('Previsões no teste (vermelho = erro)', y=1.15); plt.show()

### O mesmo modelo com as camadas prontas do PyTorch

`nn.Linear(400, 64)` guarda exatamente $\boldsymbol\Theta$ (em `.weight`) e $\boldsymbol\theta_0$ (em `.bias`). Copiando nossos parâmetros, as saídas são idênticas:

In [ ]:
import torch.nn as nn
rede_pt = nn.Sequential(nn.Linear(Di, D), nn.ReLU(), nn.Linear(D, Do))
with torch.no_grad():
    rede_pt[0].weight.copy_(Theta); rede_pt[0].bias.copy_(Theta0)
    rede_pt[2].weight.copy_(Phi);   rede_pt[2].bias.copy_(Phi0)
    print('saídas iguais?', torch.allclose(rede_pt(X_te[:100]), rede(X_te[:100]), atol=1e-5))
print(rede_pt)

---
## 12. Exercício — Problema 3.9

**Enunciado.** Mostre que a **terceira** região linear da figura 3.3 tem inclinação igual à **soma das inclinações da primeira e da quarta** regiões.

Roteiro sugerido:

1. Para cada uma das quatro regiões da figura 3.3j, identifique quais unidades estão ativas (problema 3.2).
2. Escreva a inclinação de cada região em função de $s_d = \phi_d\theta_{d1}$.
3. Conclua a relação. Por que ela vale para **quaisquer** parâmetros com a mesma configuração (mesma ordem das articulações e mesmo lado ativo de cada unidade)? Relacione com a seção 6.
4. Verifique numericamente com o código abaixo, usando os parâmetros `theta` e `phi` da seção 5. Depois mude os parâmetros (mantendo a configuração) e verifique de novo.

In [ ]:
# Complete o código.
x = np.linspace(0, 2, 2001)
y, h, pre = rede_rasa_1d(x, phi, theta)

juntas_ordenadas = None     # TODO: posições das articulações em ordem crescente
inclinacoes = None          # TODO: inclinação de cada uma das 4 regiões
                            #       (dica: np.gradient(y, x) avaliado no meio de cada região)

# print('inclinações:', inclinacoes)
# print('região 3 =', inclinacoes[2], '   região 1 + região 4 =', inclinacoes[0] + inclinacoes[3])